In [3]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [ ]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph, create_scale_free_graph, create_cycle_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop, decentralized_train_n_epochs
from src.data_utils import create_batched_dataloaders, create_dataloader

In [5]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns

In [6]:
def share_gradient(user, users, graph, item_id=None):

    commute = 0 
    commute_cost = 0
    user_neighbors = graph.adj[user.id]
    # user_gradients = {
    #     name: param.grad for name, param in user.model.named_parameters() if "item" in name
    # }
    item_parameter_names = ["item_factors.weight", "item_bias.weight"]
    if user.model_name == "gmf":
        item_parameter_names.append(f"item_layer_dict.item{item_id}.weight")
        item_parameter_names.append(f"item_layer_dict.item{item_id}.bias")
    elif user.model_name == "gmf_shared":
        item_parameter_names.append("item_layers.weight")
        item_parameter_names.append("item_layers.bias")

    user_gradients = {name: user.model.get_parameter(name).grad for name in item_parameter_names}
    # print(f"Number of elements being shared: {sum([g.numel() for _, g in user_gradients.items()])}")
    commute_cost = sum([g.numel() for _, g in user_gradients.items()])

    for neighbor_id in user_neighbors:
        neighbor = users[neighbor_id]
        neighbor_model = neighbor.model
        neighbor_optimizer = neighbor.optimizer
        neighbor_model.zero_grad() # required - diverges without this line
        commute = commute + 1
        for name in item_parameter_names:
            param = neighbor_model.get_parameter(name)
            param.grad = user_gradients[name].clone()
            
        # for name, param in neighbor_model.named_parameters():
        #     # print(name)
        #     if "item" in name:
        #         param.grad = user_gradients[name]  # Assign gradients to the neighbors
        neighbor_optimizer.step()  # Neighbors' update

    return commute, commute_cost


In [ ]:
def decentralized_train_loop(user_models, train_loader, graph, progress_bar=True):
    loss_fn = nn.MSELoss(reduction="mean")
    for _, user in user_models.items():
        user.model.train()
    losses = np.empty(len(train_loader))
    total_n_obs = 0
    total_sum_loss = 0
    avg_loss = 0
    total_commute = 0
    total_commute_cost = 0
    tbar = tqdm(train_loader) if progress_bar else train_loader
    for idx, (inputs, target) in enumerate(tbar):
        if inputs.ndim == 3:
            inputs = inputs.squeeze(0)
            target = target.squeeze(0)
        n_obs = inputs.shape[0]
        user_id = int(inputs[0, 0])
        user = user_models[user_id]
        optimizer = user.optimizer
        optimizer.zero_grad()
        score = user.model(inputs[:, 0], inputs[:, 1])
        loss = loss_fn(score, target.float())
        loss.backward()  # Calculate Gradients

        if inputs[:,1].numel() == 1:
            commute, commute_cost = share_gradient(user, user_models, graph, item_id=inputs[:,1].item())
        else:
            commute, commute_cost = share_gradient(user, user_models, graph)
        optimizer.step()  # Current user's update
        
        total_commute += commute
        total_commute_cost += commute_cost
        # print(f"Number of elements being shared: {sum([g.numel() for _, g in user_gradients.items()])}")
        total_n_obs += n_obs
        total_sum_loss += loss.detach().numpy() * n_obs
        losses[idx] = loss.detach().numpy()
        # loss_list.append(loss.detach().cpu().item())
        # avg_loss = losses[:idx + 1].mean()
        # avg_loss = total_sum_loss / total_n_obs
        avg_loss = avg_loss * (idx / (idx + 1)) + loss.detach().numpy() / (idx + 1) / n_obs
        if idx % 1000 == 0:
            if progress_bar:
                tbar.set_description(
                    f"Average Training Loss: {np.sqrt(avg_loss):.04f} | Loss: {loss.detach():.04f}"
                )
            if np.isnan(avg_loss):
                raise ValueError
    epoch_commute = {
        "number_of_commutes": total_commute,
        "commute_cost": total_commute_cost,
    }
    return avg_loss, epoch_commute
    # return avg_loss, total_commute



In [ ]:
def decentralized_train_n_epochs(user_models, train_loader, val_loader, epochs: int, graph: Graph):
    start_time = perf_counter()
    train_losses = []
    train_commutes = []
    train_commutes_cost = []
    val_losses = []
    val_loss = decentralized_validate_loop(user_models, val_loader)
    mlflow.log_metrics({"Validation RMSE": val_loss}, step=0)
    val_losses.append(val_loss)
    time_per_epoch = []

    early_stopper = EarlyStopper(patience=2, min_rel_delta=0.0001)
    for epoch in range(epochs):
        start = perf_counter()
        avg_loss, train_commute = decentralized_train_loop(
            user_models=user_models, train_loader=train_loader, graph=graph
        )

        train_losses.append(avg_loss)
        train_commutes.append(train_commute["number_of_commutes"])
        train_commutes_cost.append(train_commute["commute_cost"])
        val_loss = decentralized_validate_loop(user_models, val_loader)
        val_losses.append(val_loss)


        time_elapsed = perf_counter() - start
        time_per_epoch.append(time_elapsed)
        log_text = (
            f"Epoch {epoch + 1} | "
            f"Train Loss: {np.sqrt(avg_loss):.04f} | "
            f"Validation Loss: {val_loss:.04f} | "
            f"Time Elapsed: {time_elapsed:03f} sec |"
            f"Commute: {train_commute["number_of_commutes"]} | "
            f"Commute Cost: {train_commute["commute_cost"]}"
        )
        print(log_text)
        mlflow.log_metrics(
            {
                "Train RMSE": avg_loss,
                "Train Commute": train_commute["number_of_commutes"],
                "Train Commute Cost": train_commute["commute_cost"],
                "Validation RMSE": val_loss,
                "epochs_started": epoch+1,
            },
            step=epoch + 1,
        )
        if early_stopper.early_stop(val_loss):
            print("Early stopping.")
            break
    total_time = perf_counter() - start_time
    commute = {
        "commute": train_commutes,
        "commute_cost": train_commutes_cost,
    }
    print(f"Total time elapsed: {total_time}")
    return train_losses, val_losses, time_per_epoch, commute


In [ ]:
model = "umf"
val_loader_type = "rs"
train_loader_type = "userprop"
userprop = 0.6
n_factors = 30
sparse = False
batch_size = 10
#lr = 0.03871364416669273
#weight_decay = 0.14214480688557163
graph_seed = 1
n_epochs = 50

para_vec= {
  "scalefree_userprop" : [0.006721468985407216, 0.3793755748581348, 0.7023494584199832],
  "scalefree_urs" : [0.00797255113179729, 0.7291631699209506, 0.7649689575684868],
  "scalefree_rs" : [0.043245636749499355, 0.24293301237845355, 0.6590721600407826],
  "scalefree_oaat" : [0.014505446034196021, 0.1281494707675557, 0.3063931184178566],
    
  "cycle_userprop" : [0.03448020025507248, 0.1530360406099725, 0.3265046312442892],
  "cycle_urs" : [0.015085184891905544, 0.32597756888723617, 0.9165691479123227],
  "cycle_rs" : [0.04518354114581989, 0.07432773840871296, 0.5104116722654509],
  "cycle_oaat" : [0.006051947990064438, 0.407449910177748, 0.6941867781038726],
    
  "random2_userprop" : [0.05973335259492166, 0.20270185084925238, 0.7097401156875496],
  "random2_urs" : [0.03871364416669273, 0.14214480688557163, 0.4403378739685112],
  "random2_rs" : [0.0341364110655981, 0.006786098078128074, 0.4848527175520783],
  "random2_oaat" : [0.012098247288774554, 0.051267232285266244, 0.5034632200402083],

  "random5_userprop" : [0.01214468819649195, 0.16071055871166323, 0.8930612583507401],
  "random5_urs" : [0.04664261576162963, 0.2261414992421005, 0.3645222958218734],
  "random5_rs" : [0.01864856189846265, 0.07043500222618476, 0.850748837624225],
  "random5_oaat" : [0.004358629931177893, 0.27784542450084454, 0.41295161556157467]
    
}

In [ ]:
temp_para = para_vec["scalefree_userprop"]
lr = temp_para[0]
weight_decay = temp_para[1]
mom = temp_para[2]
print(f"lr : {lr} | wd : {weight_decay} | mom : {mom}")
["scalefree_userprop", "scalefree_urs", "scalefree_rs", "scalefree_oaat"]

In [ ]:
search_space = ["random2_userprop", "random2_urs", "random2_rs", "random2_oaat"]
model = "umf"
val_loader_type = "rs"
userprop = 0.6
n_factors = 30
sparse = False
batch_size = 50
graph_seed = 1
n_epochs = 50
test_vec = []
commute_vec = []
gtypes, dl_types = map(list, zip(*map(lambda x:x.split('_'), search_space)))  